In [44]:
import pandas as pd
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

CSV_FILENAME = "data/titanic.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (891, 12)


In [45]:
# ==========================================
# 1. MANUEL FEATURE ENGINEERING
# ==========================================
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

df['Deck'] = df['Cabin'].fillna('U').astype(str).str[0]
df['Deck'] = df['Deck'].replace('T', 'U')

features = ['Pclass','Sex','Age','Fare','FamilySize','IsAlone','Title','Deck']
X = df[features].copy()
y = df['Survived']

X['Age'] = X['Age'].fillna(X['Age'].median())
X['Fare'] = X['Fare'].fillna(X['Fare'].median())

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [46]:
# ==========================================
# 2. OTOMATİK FEATURE EXTRACTION & SELECTION İLE K-FOLD TESTİ
# ==========================================
numeric_cols = ['Age', 'Fare', 'FamilySize']

# YENİ DÜZENLEME: PolynomialFeatures SADECE sayısal sütunlara uygulanacak!
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, interaction_only=False, include_bias=False))
])

# Ön işlemci: Sayısal sütunları numeric_transformer'a sok, diğerlerine dokunma
preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, numeric_cols)],
    remainder='passthrough'
)

# Nihai Boru Hattı
pipeline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_classif, k=12)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "="*40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring='accuracy')

for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("="*40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 0.7765
Fold 2: 0.8090
Fold 3: 0.7753
Fold 4: 0.7528
Fold 5: 0.8258

Gerçek K-Fold Başarısı (Ortalama): 0.7879
Skor Sapması (Standart Sapma)    : 0.0261



In [47]:
# ==========================================
# 3. NİHAİ MODEL EĞİTİMİ VE KAYIT
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

os.makedirs('backend/models', exist_ok=True)
joblib.dump(pipeline_model, 'backend/models/titanic_pipeline.pkl')
joblib.dump(list(X_train.columns), 'backend/models/model_columns.pkl')

print("Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!")

Train Doğruluk Oranı: 0.7978
Model Doğruluk Oranı (Accuracy): 0.7765

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.82      0.79      0.81       105
           1       0.72      0.76      0.74        74

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!


In [49]:
# ==========================================
# 4. KARA KUTUYU AÇMA: Hangi Özellikler Seçildi?
# ==========================================
import pandas as pd

# 1. Pipeline içindeki adımlara (kara kutunun parçalarına) isimleriyle ulaşıyoruz
preprocessor = pipeline_model.named_steps['preprocessor']
selector = pipeline_model.named_steps['feature_selection']
classifier = pipeline_model.named_steps['classifier']

# 2. Ön işlemciden çıkan TÜM özelliklerin (Polynomial dahil) isimlerini alıyoruz
# (Scikit-Learn otomatik olarak 'num__' veya 'remainder__' gibi ön ekler koyar)
all_feature_names = preprocessor.get_feature_names_out()

# 3. SelectKBest algoritmasının seçtiklerini (True/False listesi) alıyoruz
selected_mask = selector.get_support()

# 4. Sadece seçilen (True olan) 12 özelliğin ismini filtreliyoruz
selected_features = all_feature_names[selected_mask]

# 5. Logistic Regression modelinin bu özelliklere atadığı katsayıları (ağırlıkları) çekiyoruz
coefficients = classifier.coef_[0]

# 6. Şık bir Pandas DataFrame tablosu oluşturuyoruz
feature_importance = pd.DataFrame({
    'Özellik (Feature)': selected_features,
    'Etki Ağırlığı (Coefficient)': coefficients
})

# --- GÖRSEL TEMİZLİK BÖLÜMÜ ---
# Çirkin önekleri siliyoruz
feature_importance['Özellik (Feature)'] = feature_importance['Özellik (Feature)'].str.replace('remainder__', '').str.replace('num__', '')

# Sayıları virgülden sonra 4 haneye yuvarlıyoruz
feature_importance['Etki Ağırlığı (Coefficient)'] = feature_importance['Etki Ağırlığı (Coefficient)'].round(4)
# ------------------------------

feature_importance = feature_importance.sort_values(by='Etki Ağırlığı (Coefficient)', ascending=False)

print("\n--- Modelin Seçtiği En İyi 12 Özellik ve Etkileri ---")
print(feature_importance.to_string(index=False))


--- Modelin Seçtiği En İyi 12 Özellik ve Etkileri ---
Özellik (Feature)  Etki Ağırlığı (Coefficient)
        Title_Mrs                       0.7872
           Deck_E                       0.7456
          IsAlone                       0.2812
           Fare^2                       0.0534
       Title_Miss                       0.0042
           Deck_B                      -0.0176
             Fare                      -0.1501
           Deck_C                      -0.3861
           Pclass                      -0.6706
           Deck_U                      -0.6951
         Sex_male                      -0.9742
         Title_Mr                      -1.6633
